# 01 - Preparação e Amostragem de Dados

Fluxo para carregar os dados brutos, limpar cancelados/desviados, criar features básicas e salvar uma amostra estável para as demais análises.

## Objetivos
- Carregar CSVs brutos de voos, companhias e aeroportos.
- Remover voos cancelados ou desviados que não possuem hora de chegada válida.
- Criar variáveis de atraso e tempo.
- Gerar amostra estável (`data/processed/flights_sample.parquet`) para acelerar EDA e modelagem.

In [1]:
# Imports e leitura dos dados brutos
import pandas as pd
from pathlib import Path

RAW = Path("../data/flights.csv")
AIRLINES = Path("../data/airlines.csv")
AIRPORTS = Path("../data/airports.csv")
OUT = Path("../data/processed/flights_sample.parquet")
OUT.parent.mkdir(parents=True, exist_ok=True)

### 1. Carregar dados brutos (colunas relevantes)

In [2]:
# Define colunas relevantes para carregar dos CSVs
cols = [
    "YEAR","MONTH","DAY","DAY_OF_WEEK","AIRLINE","FLIGHT_NUMBER","TAIL_NUMBER",
    "ORIGIN_AIRPORT","DESTINATION_AIRPORT","SCHEDULED_DEPARTURE","DEPARTURE_TIME",
    "DEPARTURE_DELAY","ARRIVAL_DELAY","DISTANCE","DIVERTED","CANCELLED","CANCELLATION_REASON"
]
flights_raw = pd.read_csv(RAW, usecols=cols, low_memory=False)
airlines = pd.read_csv(AIRLINES)
airports = pd.read_csv(AIRPORTS)
flights_raw.shape

(5819079, 17)

### 2. Limpeza e enriquecimento
- Remove voos cancelados (`CANCELLED = 1`) e desviados (`DIVERTED = 1`).
- Define variável-alvo `DELAYED` (>15 min).
- Cria hora de partida (`DEP_HOUR`), per?odo do dia e flag de fim de semana.

In [3]:
# Remove voos cancelados ou desviados e cria variaveis derivadas
flights = flights_raw[(flights_raw["CANCELLED"] == 0) & (flights_raw["DIVERTED"] == 0)].copy()
flights["DELAYED"] = (flights["ARRIVAL_DELAY"] > 15).astype(int)
flights["DEP_HOUR"] = (flights["SCHEDULED_DEPARTURE"] // 100).astype(int)
flights["IS_WEEKEND"] = flights["DAY_OF_WEEK"].isin([6,7]).astype(int)
flights["PERIOD_OF_DAY"] = pd.cut(
    flights["DEP_HOUR"], bins=[-1,5,11,17,23], labels=["madrugada","manh?","tarde","noite"]
)
flights[["ARRIVAL_DELAY","DELAYED","DEP_HOUR"]].head()

,ARRIVAL_DELAY,DELAYED,DEP_HOUR
0,-22.0,0,0
1,-9.0,0,0
2,5.0,0,0
3,-9.0,0,0
4,-21.0,0,0


### 3. Amostragem reprodutível
Escolhemos 300k voos para equilibrar volume e velocidade.

In [4]:
# Cria amostra reprodutivel de 300k linhas para acelerar as analises
sample = flights.sample(n=300_000, random_state=42) if len(flights) > 300_000 else flights
sample.shape

(300000, 21)

### 4. Salvar amostra processada

In [5]:
# Salva amostra processada em parquet para uso nos demais notebooks
sample.to_parquet(OUT, index=False)
print(f"Amostra salva em {OUT}")
sample.head()

Amostra salva em ..\data\processed\flights_sample.parquet


,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,...,DEPARTURE_DELAY,DISTANCE,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,DELAYED,DEP_HOUR,IS_WEEKEND,PERIOD_OF_DAY
3335058,2015,7,27,1,OO,4553,N454SW,CVG,MKE,1623,...,6.0,318,5.0,0,0,NaN,0,16,0,tarde
4857110,2015,10,30,5,DL,1615,N329NB,13204,12953,1823,...,1.0,950,-12.0,0,0,NaN,0,18,0,noite
4454180,2015,10,5,1,WN,159,N394SW,13796,14747,1150,...,8.0,672,5.0,0,0,NaN,0,11,0,manh?
5132772,2015,11,17,2,DL,1265,N942AT,ATL,SAV,1305,...,0.0,214,-3.0,0,0,NaN,0,13,0,tarde
4075015,2015,9,11,5,OO,4740,N633SK,GEG,SEA,600,...,-6.0,224,9.0,0,0,NaN,0,6,0,manh?


### 5. Notas rápidas
- `DELAYED` já usa atraso de chegada (>15 min).
- Colunas categóricas de mapeamento estão em `airlines` e `airports`.
- As demais etapas (EDA, modelagem, clusterização) consomem apenas a amostra para manter a execução leve.